# PySDK training-job status flicker — HIGH-FIDELITY reproduction

**Symptom.** In a notebook, while a training job runs, the live status panel *flickers* on every update of the elapsed-time field.

**Real code path.** `trainer` → `AgentRFTJob.wait()` → `sagemaker.train.common_utils.job_wait.wait()` → **`_wait_jupyter()`**.

**Suspected root cause.** `_wait_jupyter()` renders via a *full clear-and-redraw* each tick:
```python
clear_output(wait=True)      # tear down the whole cell output
console.print(Panel(...))    # reprint the whole panel from scratch
```

## Why this notebook is "high fidelity"
Unlike a hand-rolled panel, this notebook **imports and calls the REAL `_wait_jupyter()` from the local source tree** — unchanged. We only replace the *external I/O boundaries* with local fakes:

| Seam | Real behaviour | What we inject |
|---|---|---|
| `Job` object + `job.refresh()` | sagemaker-core resource, polls DescribeTrainingJob | `MockJob` whose fields advance from the wall clock (no AWS) |
| `job.job_config_document` | JSON with `ServiceOutput.ProgressInfo` | time-driven `CurrentStep` so the progress bar really advances |
| `_create_log_stream_handler` / `_drain_log_events` | CloudWatch Logs | fake rolling log lines (fills the *Recent Logs* section) |
| `_get_step_metrics` | MLflow REST/metrics | fake growing step-metric rows (fills the *Training Metrics* table) |
| `_is_in_studio` / `_get_studio_base_url` | reads Studio metadata file | forced on, fake base URL (adds the *Links* row) |

Everything else — the render assembly, the `iteration % 4` repaint cadence, the real `time.sleep(0.5)` loop, the `clear_output(wait=True)` + `console.print(Panel(...))` sink, `Console(force_jupyter=True)`, `_suppress_info_logging` — is the **actual production code**.

So the panel you see (height, sections, refresh rhythm) matches what a user sees, which is what makes the flicker faithful.

## How to use
1. Run **Setup**.
2. Run **Cell A (REPRO)** — calls the real `_wait_jupyter()`. Watch for flicker.
3. Run **Cell B (CANDIDATE FIX)** — runs the *same real function*, but redirects only its render sink to `rich.live.Live` (via two seam patches: `clear_output`→no-op, its `Console.print`→`Live.update`). Compare.
4. Run in **native JupyterLab** and in **Studio UI's JupyterLab** to tell whether flicker is PySDK-side (both flicker) or Studio-frontend-specific (only Studio).

## Setup

In [2]:
# IMPORTANT: run this notebook with the interpreter that has the LOCAL V3 source installed
# (the workspace's .v3test-venv). It imports the real _wait_jupyter from the local repo.
import warnings
warnings.filterwarnings("ignore")

import time
import json
from datetime import datetime, timedelta

import rich, IPython
from importlib.metadata import version
import sagemaker.train.common_utils.job_wait as jw
from sagemaker.core.resources import Unassigned
from sagemaker.train.common_utils import metrics_visualizer as mviz

print("rich", version("rich"))
print("IPython", IPython.__version__)
# Confirm we are exercising the LOCAL source, not a pip-installed sagemaker:
print("job_wait module file:", jw.__file__)
assert "sagemaker-python-sdk-lucas" in jw.__file__, (
    "Not importing the local source! Select the .v3test-venv kernel."
)
print("OK: real _wait_jupyter resolved to local source.")

rich 14.3.4
IPython 9.15.0
job_wait module file: /Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-train/src/sagemaker/train/common_utils/job_wait.py
OK: real _wait_jupyter resolved to local source.


### Mock `Job` (fields advance from the wall clock — no AWS)
Implements exactly the surface `_wait_jupyter()` touches: `job_name`, `job_arn`, `job_status`, `secondary_status`, `failure_reason`, `secondary_status_transitions`, `job_config_document`, and `refresh()`.

In [3]:
class MockTransition:
    """Mimics a sagemaker-core SecondaryStatusTransition entry.

    _calculate_transition_duration() reads .start_time/.end_time (datetimes or
    the Unassigned sentinel) and .status_message.
    """
    def __init__(self, status, start_time, end_time, status_message):
        self.status = status
        self.start_time = start_time
        self.end_time = end_time
        self.status_message = status_message


class MockJob:
    """Stand-in for sagemaker-core Job. All dynamic fields derive from elapsed
    wall-clock time, so status/progress advance with no network calls.
    """
    def __init__(self, total_seconds=45, train_start=8, max_steps=100):
        self.job_name = "flicker-repro-training-job"
        self.job_arn = (
            "arn:aws:sagemaker:us-west-2:123456789012:"
            "training-job/flicker-repro-training-job"
        )
        self.failure_reason = None
        self._start = time.time()
        self._total = total_seconds
        self._train_start = train_start
        self._max_steps = max_steps
        # Fixed transition timestamps (Starting closed, Training open/Running...)
        base = datetime.utcnow()
        self._t_starting = MockTransition(
            "Starting", base, base + timedelta(seconds=train_start),
            "Preparing the instances for training",
        )
        self._t_training = MockTransition(
            "Training", base + timedelta(seconds=train_start), Unassigned(),
            "Training image download completed. Training in progress.",
        )

    # --- refresh() is a no-op: state is derived from the clock ---
    def refresh(self, *args, **kwargs):
        return self

    @property
    def elapsed(self):
        return time.time() - self._start

    @property
    def _current_step(self):
        e = self.elapsed
        if e < self._train_start:
            return 0
        span = max(1e-6, self._total - self._train_start)
        frac = min(1.0, (e - self._train_start) / span)
        return int(frac * self._max_steps)

    @property
    def job_status(self):
        return "InProgress" if self.elapsed < self._total else "Completed"

    @property
    def secondary_status(self):
        e = self.elapsed
        if e < self._train_start:
            return "Starting"
        if e < self._total:
            return "Training"
        return "Completed"

    @property
    def secondary_status_transitions(self):
        e = self.elapsed
        if e < self._train_start:
            return [self._t_starting]
        return [self._t_starting, self._t_training]

    @property
    def job_config_document(self):
        # Real code parses this JSON each render. ProgressInfo drives the bar.
        return json.dumps({
            "ServiceOutput": {
                "ProgressInfo": {
                    "MaxSteps": self._max_steps,
                    "CurrentStep": self._current_step,
                    "BatchSize": 8,
                }
            }
            # NOTE: no MlflowConfig/MlflowDetails -> real MLflow (network) paths
            # stay disabled; we fill the metrics table via a _get_step_metrics fake.
        })


print("MockJob / MockTransition ready.")

MockJob / MockTransition ready.


### Seam fakes (logs / metrics / studio) + install helper
These fill the tall sections of the real panel *without* AWS/MLflow, so the reproduced panel matches the real one's height and section layout.

In [4]:
# Keep originals so we can restore after each run.
_ORIG = {
    "create_log": jw._create_log_stream_handler,
    "drain": jw._drain_log_events,
    "step_metrics": jw._get_step_metrics,
    "is_studio": mviz._is_in_studio,
    "studio_base": mviz._get_studio_base_url,
}

_LOG_LINES = [
    "INFO Downloading training image ...",
    "INFO Training image download completed.",
    "INFO SM_HOSTS=['algo-1']  SM_CURRENT_HOST=algo-1",
    "INFO Loading dataset shard 0 ...",
    "INFO epoch=0 step=%(s)d loss=1.842 lr=5e-6",
    "INFO epoch=0 step=%(s)d loss=1.611 lr=5e-6",
    "INFO checkpoint saved to /opt/ml/checkpoints",
]


def _fake_create_log_stream_handler(log_group, job_name, instance_count=1):
    return object()  # truthy sentinel; real code only passes it back to _drain


_log_i = {"n": 0}


def _fake_drain_log_events(handler, buf):
    # Append a couple of rolling lines per refresh cycle (deque(maxlen=20)).
    for _ in range(2):
        line = _LOG_LINES[_log_i["n"] % len(_LOG_LINES)] % {"s": _log_i["n"]}
        buf.append(line)
        _log_i["n"] += 1


def _fake_get_step_metrics(metrics_util, run_name, run_id, *args, **kwargs):
    # Grows over time -> the Training Metrics table gets taller, like a real run.
    n = min(8, 1 + _log_i["n"] // 2)
    rows = []
    for step in range(n):
        rows.append({
            "step": step,
            "rollout/reward/mean": 0.10 + 0.05 * step,
            "rollout/turns/mean": 3.0 + 0.2 * step,
            "training/total_tokens": 1024 * (step + 1),
            "training/num_trajectories": 8 * (step + 1),
        })
    return rows


def install_seam_fakes(studio=True):
    jw._create_log_stream_handler = _fake_create_log_stream_handler
    jw._drain_log_events = _fake_drain_log_events
    jw._get_step_metrics = _fake_get_step_metrics
    if studio:
        mviz._is_in_studio = lambda: True
        mviz._get_studio_base_url = lambda region: f"https://studio-d-fake.studio.{region}.sagemaker.aws"
    _log_i["n"] = 0


def restore_seams():
    jw._create_log_stream_handler = _ORIG["create_log"]
    jw._drain_log_events = _ORIG["drain"]
    jw._get_step_metrics = _ORIG["step_metrics"]
    mviz._is_in_studio = _ORIG["is_studio"]
    mviz._get_studio_base_url = _ORIG["studio_base"]


print("Seam fakes ready (install_seam_fakes / restore_seams).")

Seam fakes ready (install_seam_fakes / restore_seams).


## Cell A — REPRO (calls the REAL `_wait_jupyter`)

This invokes the actual production function. The only non-real things are the injected seams above. Timing is the real loop: `time.sleep(0.5)` with repaint on `iteration % 4` (~every 2s) and a refresh (logs/metrics/`job.refresh()`) every `poll*2` iterations.

If this flickers, the flicker lives in `_wait_jupyter`'s `clear_output(wait=True)` + `console.print(...)` sink.

In [5]:
install_seam_fakes(studio=True)
try:
    job = MockJob(total_seconds=45, train_start=8)
    jw._wait_jupyter(
        job=job,
        poll=5,                 # real default; repaint cadence ~2s, refresh ~5s
        timeout=None,
        start_time=job._start,
        log_group="/aws/sagemaker/Job/TrainingJob",
        description="flicker repro (real _wait_jupyter)",
        max_log_lines=20,
    )
finally:
    restore_seams()

╭───────────────────────────────── Agentic RFT Job Status ─────────────────────────────────╮
│  Job Name              flicker-repro-training-job                                        │
│  Job ARN               arn:aws:sagemaker:us-west-2:123456789012:training-job/flicker-re  │
│                        pro-training-job                                                  │
│  Description           flicker repro (real _wait_jupyter), 100 max steps, batch size 8   │
│  Links                 ]8;id=4247461;https://studio-d-fake.studio.us-west-2.sagemaker.aws/jobs/flicker-repro-training-job\🔗 Job (Studio)]8;;\                                                   │
│                        ]8;id=4247464;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FJob$252FTrainingJob$3FlogStreamNameFilter$3Dflicker-repro-training-job\🔗 CloudWatch Logs]8;;\                                                │
│                                                                                          │
│  Job Status            Completed                                                         │
│  Secondary Status      Completed                                                         │
│  Elapsed Time          45.2s                                                             │
│                                                                                          │
│ Status Transitions                                                                       │
│                                                                                          │
│        Step              Details                               Duration                  │
│  ───────────────────────────────────────────────────────────────────────────             │
│   ✓    Starting          Preparing the instances for           8.0s                      │
│                          training                                                        │
│   ⋯    Training          Training image download completed.    Running...                │
│                          Training in progress.                                           │
│                                                                                          │
│                                                                                          │
│ Training Metrics                                                                         │
│                                                                                          │
│                                              Training/Total     Training/Num             │
│     Step      Reward/Mean       Turns/Mean           Tokens     Trajectories             │
│  ────────────────────────────────────────────────────────────────────────────            │
│        0           0.1000                3             1024                8             │
│        1           0.1500           3.2000             2048               16             │
│        2           0.2000           3.4000             3072               24             │
│        3           0.2500           3.6000             4096               32             │
│        4           0.3000           3.8000             5120               40             │
│        5           0.3500                4             6144               48             │
│        6           0.4000           4.2000             7168               56             │
│        7           0.4500           4.4000             8192               64             │
│                                                                                          │
│                                                                                          │
│ Recent Logs (last 16)                                                                    │
│ INFO Downloading training image ...                                                      │
│ INFO Training image download completed.   

## Cell B — CANDIDATE FIX (same real function, render sink → `rich.live.Live`)

This runs the **exact same** `_wait_jupyter()` code. We only patch the two render-sink seams so the loop repaints in place instead of clear+reprint:

1. `IPython.display.clear_output` → no-op (the function imports it locally at call time, so patching the source binding takes effect).
2. the function's `Console` → a subclass whose `.print(panel)` calls `Live.update(panel)`.

Nothing else about the function changes, so this is a faithful A/B: identical content and logic, only the paint strategy differs.

In [ ]:
import IPython.display as ipd
import rich.console as rconsole
from rich.live import Live

install_seam_fakes(studio=True)

_orig_clear = ipd.clear_output
_orig_console_cls = rconsole.Console
_live_holder = {"live": None}


class _LiveRedirectConsole(_orig_console_cls):
    """Real rich Console, except .print(renderable) repaints a shared Live
    region in place instead of emitting a fresh block."""
    def print(self, *objects, **kwargs):
        live = _live_holder["live"]
        if live is not None and objects:
            live.update(objects[0])
        else:
            super().print(*objects, **kwargs)


def _noop_clear_output(*args, **kwargs):
    return None


try:
    # Patch the two render-sink seams (both are imported locally inside
    # _wait_jupyter, so patching the source modules rebinds them at call time).
    ipd.clear_output = _noop_clear_output
    rconsole.Console = _LiveRedirectConsole

    job = MockJob(total_seconds=45, train_start=8)
    with Live(refresh_per_second=8, transient=False) as live:
        _live_holder["live"] = live
        jw._wait_jupyter(
            job=job,
            poll=5,
            timeout=None,
            start_time=job._start,
            log_group="/aws/sagemaker/Job/TrainingJob",
            description="flicker repro (Live-patched sink)",
            max_log_lines=20,
        )
finally:
    ipd.clear_output = _orig_clear
    rconsole.Console = _orig_console_cls
    _live_holder["live"] = None
    restore_seams()